# Sky Trails — ADS-B Data Router Prototype

This notebook collects live ADS-B aircraft messages from the websocket stream.

The goal is to turn individual aircraft packets into clean trail points that can later be used for our map visualization.

## 1. Imports

This section loads the Python libraries we need.

- `connect` lets us connect to the live websocket stream.
- `json` lets us read and write JSON data.

In [ ]:
from websockets.sync.client import connect
import json

In this example we use the websockets library (https://websockets.readthedocs.io/en/stable/)

This is a super simple way to receive the websocket data, but feel free to use other methods

## 2. Settings

These settings control the main parts of the notebook.

Keeping them here makes it easier to change the websocket address, the number of messages we collect, and the output file name without searching through the code.

In [ ]:
ADSB_WEBSOCKET_URL = "ws://192.87.172.82:1338"

MAX_MESSAGES = 1000

OUTPUT_FILE = "trail_points.json"

## 3. Packet Format

The websocket gives us one JSON entry for each ADS-B packet received by one of the campus receivers.

A single packet does **not** always contain all aircraft information.
This is why we combine packets with the same aircraft `address` later in the notebook.

### 3.1 Possible fields

| Field | Meaning |
|---|---|
| `address` | The aircraft address. We use this as the plane ID. |
| `altitude` | The aircraft altitude, in feet. |
| `latitude` | The north/south map position of the aircraft. |
| `longitude` | The east/west map position of the aircraft. |
| `speed` | The aircraft speed, likely in knots. |
| `heading` | The direction the aircraft is moving, in degrees. |
| `callsign` | The flight/aircraft callsign, when available. |
| `timestamp` | The time when the receiver received the packet. |
| `rssi` | Signal strength received by the antenna. |
| `receiver` | The receiver that heard the packet, either `zi-5067` or `zi-5110`. |

### 3.2 Two receivers

Because there are two receivers, the same aircraft packet may sometimes appear twice.

When this happens, most aircraft fields may be identical, but fields such as `timestamp`, `rssi`, and `receiver` can differ. This is useful for comparing how each receiver hears the same aircraft.

### 3.3 Different packet types

There are different kinds of ADS-B messages. Each kind gives us a different puzzle piece.

| Message type | Usually contains |
|---|---|
| Location message | `latitude` and `longitude` |
| Altitude message | `altitude` |
| Speed message | `speed` and `heading` |
| Callsign message | `callsign` |

Because these pieces can arrive separately, the notebook uses `plane_memory` to remember the latest known information for each aircraft.

## 4. Data Storage

Before we receive live data, we create two empty containers where useful aircraft information can be stored.

We use two containers:

- `plane_memory` remembers the latest known information for each aircraft.
- `trail_points` stores drawable map points over time.

In [ ]:
plane_memory = {}
trail_points = []

## 5. Message Handling

This function decides what to do with one incoming ADS-B message.

It uses the aircraft `address` to update `plane_memory`.
If the message contains a drawable position, it also saves a point into `trail_points`.

In simple words: this is where one small packet becomes part of a plane trail.

In [ ]:
def handle_message(msg, seen_addresses):
    address = msg.get("address")

    # If the message has no aircraft address, we cannot group it.
    if address is None:
        return

    # If this is the first time we see this aircraft,
    # create a little memory entry for it.
    if address not in plane_memory:
        plane_memory[address] = {
            "address": address,
            "speed": None,
            "heading": None,
            "altitude": None,
            "latitude": None,
            "longitude": None,
            "timestamp": None,
            "rssi": None,
            "receiver": None
        }

    # Update only the fields that are present in this message.
    for field in [
        "speed",
        "heading",
        "altitude",
        "latitude",
        "longitude",
        "timestamp",
        "rssi",
        "receiver"
    ]:
        if field in msg:
            plane_memory[address][field] = msg[field]

    # Only print planes that are useful for drawing a map.
    has_position = (
        plane_memory[address]["latitude"] is not None
        and plane_memory[address]["longitude"] is not None
    )

    if has_position:
        point = {
            "address": address,
            "latitude": plane_memory[address]["latitude"],
            "longitude": plane_memory[address]["longitude"],
            "altitude": plane_memory[address]["altitude"],
            "speed": plane_memory[address]["speed"],
            "heading": plane_memory[address]["heading"],
            "timestamp": plane_memory[address]["timestamp"],
            "rssi": plane_memory[address]["rssi"],
            "receiver": plane_memory[address]["receiver"]
        }

        trail_points.append(point)

    if has_position and address not in seen_addresses:
        print("Plane memory now has a drawable plane:")
        print(json.dumps(plane_memory[address], indent=2))
        print()
        seen_addresses.add(address)

## 6. Receive Live ADS-B Data

This function opens the websocket connection and listens for incoming ADS-B packets.

For each message, it:
1. receives a raw websocket message,
2. turns it from JSON text into a Python dictionary,
3. sends it to `handle_message()` for processing,
4. repeats this until `MAX_MESSAGES` messages have been received.

This function is the “river gate” of the notebook: it lets live aircraft data flow into our processing code.

If the websocket stops sending messages, the function ends gracefully and keeps the data collected so far.

In [ ]:
def receive_data():
    """
    Connect to the ADS-B websocket and collect live aircraft messages.

    The function stops when it reaches MAX_MESSAGES.
    If the websocket pauses or closes, it keeps the data collected so far.
    """

    max_msg = MAX_MESSAGES

    # Keeps track of which aircraft we have already printed during this run.
    # This prevents the notebook from printing the same aircraft too many times.
    seen_addresses = set()

    with connect(ADSB_WEBSOCKET_URL) as websocket:
        count = 0

        while count < max_msg:
            try:
                raw_msg = websocket.recv()
                msg = json.loads(raw_msg)

                handle_message(msg, seen_addresses)

                count += 1

            except TimeoutError:
                print("No new message arrived for a while. Stopping collection calmly.")
                break

            except json.JSONDecodeError:
                print("Failed to decode JSON. Skipping this packet and continuing...")

            except Exception as error:
                print("The websocket stopped or something unexpected happened.")
                print("Keeping the data collected so far.")
                print("Error:", error)
                break

        print(f"Received {count} messages!")

## 7. Run Data Collection

This cell starts the live data collection process.

It calls `receive_data()`, which connects to the ADS-B websocket, receives messages, and sends each message to `handle_message()`.

After this cell runs, `plane_memory` and `trail_points` should contain collected aircraft data.

In [ ]:
receive_data()

## 8. Inspect Collected Trail Points

This section checks what we collected after running the live data collection.

`len(trail_points)` tells us how many drawable map points we saved.
`trail_points[:3]` shows the first three points, so we can quickly check whether the data looks correct.

In [ ]:
len(trail_points)
trail_points[:3]

## 9. Save Trail Points for Visualization

The collected trail points currently live only in notebook memory.

This step saves them into a JSON file, so the visualization part of the project can load the data without needing to connect to the live websocket every time.

In [ ]:
# Save collected trail points as a JSON file.
# This turns our temporary notebook data into a reusable file.

with open(OUTPUT_FILE, "w") as file:
    json.dump(trail_points, file, indent=2)

print(f"Saved {len(trail_points)} trail points to {OUTPUT_FILE}")

## 10. Check Saved Trail Points

This section reads the saved JSON file back into Python.

If this works, we know that `trail_points.json` was created correctly and can be used by the visualization part of the project.

This is a small safety check before moving on to the map.

In [ ]:
# Read the saved trail points back from the JSON file.
# This checks whether the file was created correctly.

with open(OUTPUT_FILE, "r") as file:
    saved_trail_points = json.load(file)

print(f"Loaded {len(saved_trail_points)} trail points from {OUTPUT_FILE}")

saved_trail_points[:3]

## 11. First Map Prototype

This section creates a simple interactive map from the saved trail points.

For now, each aircraft position is shown as a dot. Later, we can connect dots with the same `address` to create trails.

In [ ]:
# Create a first simple map from the saved trail points.

import folium

# Keep only points that have a valid map position.
valid_points = [
    point for point in saved_trail_points
    if point["latitude"] is not None and point["longitude"] is not None
]

# Use the average position as the center of the map.
center_latitude = sum(point["latitude"] for point in valid_points) / len(valid_points)
center_longitude = sum(point["longitude"] for point in valid_points) / len(valid_points)

plane_map = folium.Map(
    location=[center_latitude, center_longitude],
    zoom_start=7
)

# Add a dot for each saved aircraft position.
for point in valid_points:
    popup_text = f"""
    Aircraft: {point["address"]}<br>
    Altitude: {point["altitude"]}<br>
    Speed: {point["speed"]}<br>
    Heading: {point["heading"]}<br>
    Receiver: {point["receiver"]}<br>
    RSSI: {point["rssi"]}
    """

    folium.CircleMarker(
        location=[point["latitude"], point["longitude"]],
        radius=4,
        popup=popup_text,
        fill=True
    ).add_to(plane_map)

plane_map.save("first_plane_map.html")

print("Saved map to first_plane_map.html")

plane_map

## 12. Group Trail Points by Aircraft

This section groups saved trail points by aircraft `address`.

A trail can only be drawn when the same aircraft has at least two position points.

In [ ]:
# Group trail points by aircraft address.
# Each aircraft will get its own list of map points.

points_by_aircraft = {}

for point in saved_trail_points:
    address = point["address"]

    if address not in points_by_aircraft:
        points_by_aircraft[address] = []

    points_by_aircraft[address].append(point)

print(f"Found {len(points_by_aircraft)} different aircraft.")

for address, points in points_by_aircraft.items():
    print(address, "has", len(points), "points")

## 13. Altitude-Based Trail Colors

This helper function chooses a trail color based on aircraft altitude.

For this first version, each trail uses the aircraft's last known altitude.
Lower aircraft get warmer sunset colors, while higher aircraft get cooler moonlit sky colors.

In [ ]:
def get_altitude_color(altitude):
    """
    Choose a soft sky-inspired color based on altitude in feet.

    Lower aircraft use warmer colors.
    Higher aircraft use cooler moonlit colors.
    """

    if altitude is None:
        return "#B8B8B8"  # soft gray for unknown altitude

    if altitude < 10000:
        return "#F6A97A"  # warm peach: lower / closer to earth

    if altitude < 25000:
        return "#F3A6B8"  # soft pink: middle sky

    if altitude < 35000:
        return "#B8A7E8"  # lavender: cruising altitude

    return "#6F8DEB"      # moon blue: very high altitude

## 14. Map Legend Helper

This helper adds a small legend to the map.

The legend explains how trail colors relate to aircraft altitude.

In [ ]:
def add_altitude_legend(map_object):
    """
    Add an altitude color legend to a Folium map.

    The legend explains the dreamy sky colors used for aircraft trails.
    """

    legend_html = """
    <div style="
        position: fixed;
        bottom: 40px;
        left: 40px;
        z-index: 9999;
        background-color: white;
        padding: 12px;
        border: 2px solid #888;
        border-radius: 10px;
        font-size: 14px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.25);
    ">
        <b>Trail altitude</b><br>

        <div>
            <span style="background-color:#F6A97A; width:14px; height:14px; display:inline-block; margin-right:6px;"></span>
            Below 10,000 ft
        </div>

        <div>
            <span style="background-color:#F3A6B8; width:14px; height:14px; display:inline-block; margin-right:6px;"></span>
            10,000–25,000 ft
        </div>

        <div>
            <span style="background-color:#B8A7E8; width:14px; height:14px; display:inline-block; margin-right:6px;"></span>
            25,000–35,000 ft
        </div>

        <div>
            <span style="background-color:#6F8DEB; width:14px; height:14px; display:inline-block; margin-right:6px;"></span>
            Above 35,000 ft
        </div>

        <div>
            <span style="background-color:#B8B8B8; width:14px; height:14px; display:inline-block; margin-right:6px;"></span>
            Unknown altitude
        </div>
    </div>
    """

    map_object.get_root().html.add_child(folium.Element(legend_html))

## 15. First Aircraft Trails

This section draws simple aircraft trails.

Each aircraft gets one line if it has at least two saved position points.
Aircraft with only one point are skipped for now because one point cannot form a trail.

In [ ]:
# Create a new map that shows aircraft trails instead of only dots.

trail_map = folium.Map(
    location=[center_latitude, center_longitude],
    zoom_start=7
)

for address, points in points_by_aircraft.items():

    # A trail needs at least two points.
    if len(points) < 2:
        continue

    # Sort points by timestamp so the trail follows time order.
    points = sorted(points, key=lambda point: point["timestamp"])

    # Folium expects coordinates as [latitude, longitude].
    coordinates = [
        [point["latitude"], point["longitude"]]
        for point in points
    ]

    # Create a small popup summary for this aircraft.
    first_point = points[0]
    last_point = points[-1]

    trail_color = get_altitude_color(last_point["altitude"])

    popup_text = f"""
    Aircraft: {address}<br>
    Points in trail: {len(points)}<br>
    First timestamp: {first_point["timestamp"]}<br>
    Last timestamp: {last_point["timestamp"]}<br>
    Altitude: {last_point["altitude"]} ft<br>
    Last receiver: {last_point["receiver"]}
    """

    folium.PolyLine(
        locations=coordinates,
        popup=popup_text,
        color=trail_color,
        weight=8
    ).add_to(trail_map)

add_altitude_legend(trail_map)

trail_map.save("first_aircraft_trails.html")

print("Saved trail map to first_aircraft_trails.html")

trail_map

## 16. Segment-Colored Aircraft Trails

This section draws aircraft trails as smaller line segments.

Each segment is colored based on the aircraft altitude at that part of the trail.
This makes altitude changes visible along the path instead of giving the whole aircraft trail one color.

In [ ]:
# Create a new map where each aircraft trail is made from smaller colored segments.

segment_trail_map = folium.Map(
    location=[center_latitude, center_longitude],
    zoom_start=7
)

for address, points in points_by_aircraft.items():

    # A segmented trail still needs at least two points.
    if len(points) < 2:
        continue

    # Sort points by timestamp so the trail follows time order.
    points = sorted(points, key=lambda point: point["timestamp"])

    # Draw one small line segment between each pair of neighboring points.
    for index in range(len(points) - 1):
        start_point = points[index]
        end_point = points[index + 1]

        start_location = [start_point["latitude"], start_point["longitude"]]
        end_location = [end_point["latitude"], end_point["longitude"]]

        # Skip segments where the aircraft position did not visibly change.
        if start_location == end_location:
            continue

        # Use the altitude at the end of the segment to choose the color.
        segment_color = get_altitude_color(end_point["altitude"])

        popup_text = f"""
        Aircraft: {address}<br>
        Segment: {index + 1}<br>
        Altitude: {end_point["altitude"]} ft<br>
        Speed: {end_point["speed"]}<br>
        Heading: {end_point["heading"]}<br>
        Receiver: {end_point["receiver"]}<br>
        RSSI: {end_point["rssi"]}<br>
        Timestamp: {end_point["timestamp"]}
        """

        folium.PolyLine(
            locations=[start_location, end_location],
            popup=popup_text,
            color=segment_color,
            weight=6,
            opacity=0.85
        ).add_to(segment_trail_map)

add_altitude_legend(segment_trail_map)

segment_trail_map.save("segment_colored_aircraft_trails.html")

print("Saved segment-colored trail map to segment_colored_aircraft_trails.html")

segment_trail_map

## 17. Relative Altitude Gradient Colors

This helper creates a smoother altitude color gradient based on the altitude range in the current saved dataset.

This is useful for the prototype because it makes altitude differences more visible, even when the collection time is short.

### 17.1 Find the Altitude Range

Before we can create a relative gradient, we need to know the lowest and highest altitude in the saved data.

These values become the endpoints of the gradient:
- lowest altitude → warm peach
- highest altitude → moon blue

In [ ]:
# Find the altitude range in the current saved data.
# This lets us color trails relative to this specific dataset.

altitudes = [
    point["altitude"]
    for point in saved_trail_points
    if point["altitude"] is not None
]

MIN_ALTITUDE = min(altitudes)
MAX_ALTITUDE = max(altitudes)

print(f"Lowest altitude in this dataset: {MIN_ALTITUDE} ft")
print(f"Highest altitude in this dataset: {MAX_ALTITUDE} ft")

### 17.2 Gradient Helper Functions

These functions turn an altitude into a smooth color.

The small blending functions handle the math for mixing colors, while `get_relative_altitude_color()` applies that blending to aircraft altitude.

In [ ]:
def blend_channel(start_value, end_value, amount):
    """
    Blend one RGB color channel.

    amount = 0 gives the start value.
    amount = 1 gives the end value.
    """

    return round(start_value + (end_value - start_value) * amount)


def blend_color(start_color, end_color, amount):
    """
    Blend between two RGB colors and return a hex color.
    """

    red = blend_channel(start_color[0], end_color[0], amount)
    green = blend_channel(start_color[1], end_color[1], amount)
    blue = blend_channel(start_color[2], end_color[2], amount)

    return f"#{red:02X}{green:02X}{blue:02X}"


def get_relative_altitude_color(altitude):
    """
    Choose a smooth sky-inspired color based on this dataset's altitude range.

    The lowest altitude becomes warm peach.
    The middle altitude becomes lavender.
    The highest altitude becomes moon blue.
    """

    if altitude is None:
        return "#B8B8B8"  # soft gray for unknown altitude

    if MAX_ALTITUDE == MIN_ALTITUDE:
        return "#B8A7E8"  # fallback if all altitudes are the same

    # Convert altitude into a 0–1 position inside the dataset's altitude range.
    altitude_ratio = (altitude - MIN_ALTITUDE) / (MAX_ALTITUDE - MIN_ALTITUDE)

    # Keep the ratio safely between 0 and 1.
    altitude_ratio = max(0, min(1, altitude_ratio))

    peach = (246, 169, 122)
    lavender = (184, 167, 232)
    moon_blue = (111, 141, 235)

    # First half: peach → lavender
    if altitude_ratio < 0.5:
        local_ratio = altitude_ratio / 0.5
        return blend_color(peach, lavender, local_ratio)

    # Second half: lavender → moon blue
    local_ratio = (altitude_ratio - 0.5) / 0.5
    return blend_color(lavender, moon_blue, local_ratio)

### 17.3 Relative Gradient Legend Helper

This helper adds a legend for the relative altitude gradient.

Unlike the earlier category legend, this one explains the color range using the lowest and highest altitudes found in the current saved dataset.

In [ ]:
def add_relative_altitude_legend(map_object):
    """
    Add a relative altitude gradient legend to a Folium map.

    The legend shows that colors are based on the current dataset:
    lowest altitude = warm peach
    middle altitude = lavender
    highest altitude = moon blue
    """

    legend_html = f"""
    <div style="
        position: fixed;
        bottom: 40px;
        left: 40px;
        z-index: 9999;
        background-color: white;
        padding: 12px;
        border: 2px solid #888;
        border-radius: 10px;
        font-size: 14px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.25);
        min-width: 220px;
    ">
        <b>Relative trail altitude</b><br>

        <div style="
            height: 14px;
            margin-top: 8px;
            margin-bottom: 6px;
            border-radius: 8px;
            background: linear-gradient(to right, #F6A97A, #B8A7E8, #6F8DEB);
        "></div>

        <div style="display: flex; justify-content: space-between; font-size: 12px;">
            <span>{MIN_ALTITUDE} ft</span>
            <span>{MAX_ALTITUDE} ft</span>
        </div>

        <div style="font-size: 12px; margin-top: 6px;">
            Colors are scaled to this saved dataset.
        </div>
    </div>
    """

    map_object.get_root().html.add_child(folium.Element(legend_html))

## 18. Relative Gradient Segment Trails

This section draws segmented aircraft trails using the relative altitude gradient.

Each small trail segment gets a color based on its altitude compared to the lowest and highest altitude in the current saved dataset.

In [ ]:
# Create a map where each aircraft trail is made from small segments.
# Each segment uses the relative altitude gradient color.

relative_segment_map = folium.Map(
    location=[center_latitude, center_longitude],
    zoom_start=7
)

for address, points in points_by_aircraft.items():

    # A segmented trail needs at least two points.
    if len(points) < 2:
        continue

    # Sort points by timestamp so the trail follows time order.
    points = sorted(points, key=lambda point: point["timestamp"])

    # Draw one small line segment between each pair of neighboring points.
    for index in range(len(points) - 1):
        start_point = points[index]
        end_point = points[index + 1]

        start_location = [start_point["latitude"], start_point["longitude"]]
        end_location = [end_point["latitude"], end_point["longitude"]]

        # Skip segments where the aircraft position did not visibly change.
        if start_location == end_location:
            continue

        # Use the relative gradient color for this segment's altitude.
        segment_color = get_relative_altitude_color(end_point["altitude"])

        popup_text = f"""
        Aircraft: {address}<br>
        Segment: {index + 1}<br>
        Altitude: {end_point["altitude"]} ft<br>
        Speed: {end_point["speed"]}<br>
        Heading: {end_point["heading"]}<br>
        Receiver: {end_point["receiver"]}<br>
        RSSI: {end_point["rssi"]}<br>
        Timestamp: {end_point["timestamp"]}
        """

        folium.PolyLine(
            locations=[start_location, end_location],
            popup=popup_text,
            color=segment_color,
            weight=7,
            opacity=0.9
        ).add_to(relative_segment_map)

add_relative_altitude_legend(relative_segment_map)

relative_segment_map.save("relative_gradient_segment_trails.html")

print("Saved relative gradient segment map to relative_gradient_segment_trails.html")

relative_segment_map

## 19. Reusable Relative-Gradient Map Function

This section turns the relative-gradient trail map into a reusable function.

This is important for the future live version, because the program will need to redraw and save the map repeatedly as new ADS-B packets arrive.

### 19.1 Dynamic Gradient Helper

This helper chooses a relative altitude color using a given minimum and maximum altitude.

Unlike the earlier version, it does not depend on fixed global `MIN_ALTITUDE` and `MAX_ALTITUDE` values. This makes it safer for live-updating maps.

In [20]:
def get_relative_altitude_color_for_range(altitude, min_altitude, max_altitude):
    """
    Choose a smooth sky-inspired color using a specific altitude range.

    This version is useful for live-updating maps because the altitude
    range can be recalculated each time the map is redrawn.
    """

    if altitude is None:
        return "#B8B8B8"  # soft gray for unknown altitude

    if max_altitude == min_altitude:
        return "#B8A7E8"  # fallback if all altitudes are the same

    # Convert altitude into a 0–1 position inside the current altitude range.
    altitude_ratio = (altitude - min_altitude) / (max_altitude - min_altitude)
    altitude_ratio = max(0, min(1, altitude_ratio))

    peach = (246, 169, 122)
    lavender = (184, 167, 232)
    moon_blue = (111, 141, 235)

    # First half: peach → lavender
    if altitude_ratio < 0.5:
        local_ratio = altitude_ratio / 0.5
        return blend_color(peach, lavender, local_ratio)

    # Second half: lavender → moon blue
    local_ratio = (altitude_ratio - 0.5) / 0.5
    return blend_color(lavender, moon_blue, local_ratio)

### 19.2 Dynamic Legend Helper

This helper adds a relative altitude legend using the current minimum and maximum altitude.

The legend can therefore update when the dataset changes.

In [21]:
def add_relative_altitude_legend_for_range(map_object, min_altitude, max_altitude):
    """
    Add a relative altitude gradient legend to a Folium map.

    The legend uses the altitude range passed into the function,
    which makes it suitable for maps that are redrawn repeatedly.
    """

    legend_html = f"""
    <div style="
        position: fixed;
        bottom: 40px;
        left: 40px;
        z-index: 9999;
        background-color: white;
        padding: 12px;
        border: 2px solid #888;
        border-radius: 10px;
        font-size: 14px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.25);
        min-width: 220px;
    ">
        <b>Relative trail altitude</b><br>

        <div style="
            height: 14px;
            margin-top: 8px;
            margin-bottom: 6px;
            border-radius: 8px;
            background: linear-gradient(to right, #F6A97A, #B8A7E8, #6F8DEB);
        "></div>

        <div style="display: flex; justify-content: space-between; font-size: 12px;">
            <span>{min_altitude} ft</span>
            <span>{max_altitude} ft</span>
        </div>

        <div style="font-size: 12px; margin-top: 6px;">
            Colors are scaled to the current trail data.
        </div>
    </div>
    """

    map_object.get_root().html.add_child(folium.Element(legend_html))

### 19.3 Reusable Map Builder

This function creates a relative-gradient segmented aircraft trail map from a list of trail points.

It groups points by aircraft, calculates the current altitude range, draws colored trail segments, adds a legend, saves the map as HTML, and returns the map object.

In [22]:
def create_relative_gradient_segment_map(
    trail_points_data,
    output_html_file="live_aircraft_trails.html",
    zoom_start=7,
    line_weight=8
):
    """
    Create a Folium map with segmented aircraft trails.

    Each segment is colored using a relative altitude gradient based on
    the minimum and maximum altitude in the given trail point data.
    """

    # Keep only points that have valid map coordinates.
    valid_points = [
        point for point in trail_points_data
        if point["latitude"] is not None and point["longitude"] is not None
    ]

    if len(valid_points) == 0:
        print("No valid map points available yet.")
        return None

    # Use the average location as the map center.
    center_latitude = sum(point["latitude"] for point in valid_points) / len(valid_points)
    center_longitude = sum(point["longitude"] for point in valid_points) / len(valid_points)

    # Collect altitudes for the relative gradient.
    altitudes = [
        point["altitude"]
        for point in valid_points
        if point["altitude"] is not None
    ]

    if len(altitudes) == 0:
        min_altitude = 0
        max_altitude = 0
    else:
        min_altitude = min(altitudes)
        max_altitude = max(altitudes)

    # Group points by aircraft address.
    points_by_aircraft = {}

    for point in valid_points:
        address = point["address"]

        if address not in points_by_aircraft:
            points_by_aircraft[address] = []

        points_by_aircraft[address].append(point)

    # Create the map.
    map_object = folium.Map(
        location=[center_latitude, center_longitude],
        zoom_start=zoom_start
    )

    # Draw segmented trails.
    for address, points in points_by_aircraft.items():

        if len(points) < 2:
            continue

        points = sorted(points, key=lambda point: point["timestamp"])

        for index in range(len(points) - 1):
            start_point = points[index]
            end_point = points[index + 1]

            start_location = [start_point["latitude"], start_point["longitude"]]
            end_location = [end_point["latitude"], end_point["longitude"]]

            # Skip duplicate positions to avoid drawing invisible tiny segments.
            if start_location == end_location:
                continue

            segment_color = get_relative_altitude_color_for_range(
                end_point["altitude"],
                min_altitude,
                max_altitude
            )

            popup_text = f"""
            Aircraft: {address}<br>
            Segment: {index + 1}<br>
            Altitude: {end_point["altitude"]} ft<br>
            Speed: {end_point["speed"]}<br>
            Heading: {end_point["heading"]}<br>
            Receiver: {end_point["receiver"]}<br>
            RSSI: {end_point["rssi"]}<br>
            Timestamp: {end_point["timestamp"]}
            """

            folium.PolyLine(
                locations=[start_location, end_location],
                popup=popup_text,
                color=segment_color,
                weight=line_weight,
                opacity=0.9
            ).add_to(map_object)

    add_relative_altitude_legend_for_range(
        map_object,
        min_altitude,
        max_altitude
    )

    map_object.save(output_html_file)

    print(f"Saved map to {output_html_file}")
    print(f"Used {len(valid_points)} valid trail points.")
    print(f"Found {len(points_by_aircraft)} aircraft.")
    print(f"Altitude range: {min_altitude} ft — {max_altitude} ft")

    return map_object

### 19.4 Test the Reusable Map Builder

This cell checks whether the reusable map function can recreate the relative-gradient trail map from the saved trail points.

If this works, we know the map-building logic is ready to be used later inside an auto-updating loop.

In [23]:
test_live_map = create_relative_gradient_segment_map(
    saved_trail_points,
    output_html_file="test_reusable_relative_gradient_map.html",
    zoom_start=7,
    line_weight=7
)

test_live_map

Saved map to test_reusable_relative_gradient_map.html
Used 435 valid trail points.
Found 11 aircraft.
Altitude range: 19425 ft — 39000 ft


## Later Improvements

These are ideas we may want to improve later, but they are not needed for the current prototype.

- Make the altitude legend size adjustable or more responsive.
- Add a cleaner final visual style for the exported HTML map.
- Consider making trail thickness or opacity show signal strength (`rssi`).
- Consider using different line styles for different receivers.
- Add food/on-board service data to the aircraft popup once the food database is ready.
- Improve aircraft turning/path detail by collecting more messages over a longer time window, so segmented trails can show smoother curves and direction changes.
- Upgrade the auto-refresh prototype into a real live web app where the browser map updates continuously as new ADS-B packets arrive.